# FIJI ROI Test 
----

Ximena has generated some ROI's in Fiji, the test now is can I use python to open these ROIs and apply them to one of the images.  

------
### Import dependancies

In [1]:
import numpy as np 
import skimage 
import scipy 

import os 
import pandas as pd

import tkinter as tk 
from tkinter import filedialog

import tifffile as tf
import matplotlib.pyplot as plt

import roifile

from tqdm import tqdm

---- 
### Select the applicable ROI files

In [2]:
def get_all_roi_files(top_folder, folders, file):
    ''' '''
    #
    roi_folder = []
    # 
    for i in range( len(folders) ):
        #
        if folders[i].find(file[0:-4]) > -1: 
            # 
            roi_folder.append(top_folder + '/' + folders[i] + '/')
    #
    if len(roi_folder) == 0: 
        return([])
    
    else:
        #
        roi_files = os.listdir(roi_folder[0])
        #
        file_names = []
        
        for j in range( len(roi_files) ):
            # 
            file_names.append(roi_folder[0] + roi_files[j])
        
        return(file_names)

----
### Get ROI file data 

In [3]:
def roi_retrieval(roi_file): 

    # initalise 
    roi = roifile.roiread(roi_file)
    # Convert ROI data to np array. 
    roi_polygon = np.array(roi.coordinates())

    return(roi_polygon)

In [4]:
def create_crops_and_peak_find(polygon, image, time):
    ''' '''

    #
    cell_square_extremes = np.array([np.min(polygon[:,0]), np.max(polygon[:,0]), 
                       np.min(polygon[:,1]), np.max(polygon[:,1])], dtype=int)
    #
    crop = image[time, cell_square_extremes[2]:cell_square_extremes[3], 
                        cell_square_extremes[0]:cell_square_extremes[1]]
    
    return(crop)
    

In [5]:
def calculate_distances(peaks, crop): 

    # Find image centre
    centre = [int( crop.shape[0] / 2) , int( crop.shape[1] / 2) ]
    # Storage variables
    x_distances = []
    y_distances = []
    
    # For all peaks
    for i in range( len( peaks) ):
        # Calculate the distance of the peak from the 
        # centre of the image crop 
        # (and therefore hopefully the cell)
        x_distances = centre[0] - peaks[i][0]
        y_distances = centre[1] - peaks[i][1]
    # Calculate the mean peak positions
    mean_x_distance = np.sum(x_distances)
    mean_y_distance = np.sum(y_distances)

    return(mean_x_distance, mean_y_distance)

----
### Filter the masks

In [6]:
def size_filter(mask_label):
    ''' '''
    # initalise
    props = {'label', 'area'}
    new_masks = np.zeros(mask_label.shape)
    new_mask_val = 1

    # Find the mask properties
    mask_props = pd.DataFrame( skimage.measure.regionprops_table(
            mask_label, properties = props ) )

    # For all masks in array
    for i in range( mask_props.shape[0] ): 
        # If mask area is more than 10 pixels. 
        if mask_props['area'].iloc[i] > 10: 
            # Find the pixels where the mask exist. 
            mask_loc = np.where( mask_label == mask_props['label'].iloc[i] )
            # Keep the mask above 10 pixels. 
            new_masks[mask_loc[0], mask_loc[1]] = new_mask_val
            # Plus 1 to counter 
            new_mask_val = new_mask_val + 1

    # return new mask shape
    return( new_masks )

---- 
### Find peaks in each mask

In [7]:
def peaks_in_masks(mask_labels, image_crop, threshold): 
    ''' '''
    # Initalise
    peak_list = pd.DataFrame()
    
    # 
    for i in range( int(np.max( mask_labels ) ) ):
        #
        crop_double = np.array( mask_labels )
        # 
        mask_id = i + 1 
        # 
        crop_double[crop_double != mask_id] = 0 
        crop_double[crop_double == mask_id] = 1
        #
        max_peaks = np.array(skimage.feature.peak_local_max( crop_double * image_crop, 
                                    threshold_abs = threshold,  min_distance = 5) )
        
        peak_dict = {'x':max_peaks[:, 0], 'y':max_peaks[:,1] }

        peak_list = pd.concat([peak_list, pd.DataFrame( peak_dict )])

    return( peak_list )

In [13]:
def save_figures(im_crop, mask_im, peak_points, save_loc, file_name, time): 

    fig1, ax1 = plt.subplots(figsize = (10, 10))
    # setting font sizeto 30
    plt.rcParams.update({'font.size': 14})
    ax1.imshow(im_crop)
    # plt.colorbar(h, shrink = 0.8, label = '\nImage Intensity')
    fig1.savefig(save_loc + file_name + '_t' + str(time) + 
                '_image_data.tif')
    
    fig2, ax2= plt.subplots(figsize = (10, 10))
    # setting font sizeto 30
    plt.rcParams.update({'font.size': 14})
    ax2.imshow(im_crop)
    ax2.imshow(mask_im, alpha = 0.5)
    fig2.savefig(save_loc + file_name + '_t' + str(time) + 
                '_image_data_and_masks.tif')

    if len(peak_points) > 0:
        fig3, ax3= plt.subplots(figsize = (10, 10))
        # setting font sizeto 30
        plt.rcParams.update({'font.size': 14})
        ax3.imshow(im_crop)
        ax3.imshow(mask_im, alpha = 0.5)
        ax3.scatter(peak_points['y'], peak_points['x'], 
                    marker = 'x', color = 'r')
        fig3.savefig(save_loc + file_name + '_t' + str(time) + 
                    '_image_data_and_masks_and_peaks.tif' )

    tf.imwrite(save_loc + 'T_' + str(time) + '_' + file_name[0:-4] + '.tif', 
              im_crop)

----
### Select image data 

In [16]:
# # # Creates dialogue to ask directory
# # # Get the folder containing the image stack. 
root = tk.Tk()
root.attributes('-topmost', True)
root.withdraw() # Stops a second window opening
folder = filedialog.askdirectory(title = 'Select Stack file')

print(folder)

//data.beatson.gla.ac.uk/data/RCORBYN/User_Data/Current_Projects/20250611_Ximena_ReceptorSegregation/XR_198_MIP_Images


----
### Image Analysis

In [17]:
# Get a list of files form the folder
files = os.listdir(folder)

# initlaise 
peak_df = pd.DataFrame()
save_path = os.path.dirname(folder) + '/figures/'
if os.path.exists(save_path) == False: 
    os.makedirs(save_path)

# Select the image 
roi_top_folder = '//data.beatson.gla.ac.uk/data/RCORBYN/User_Data/Current_Projects/20250611_Ximena_ReceptorSegregation/XR198_199_postFIJI/'
roi_folders = os.listdir(roi_top_folder)

#
for file in tqdm( files ): 
    # Get image data
    im_data = tf.imread( folder + '/' + file )
    roi_files = get_all_roi_files(roi_top_folder, roi_folders, file)
    # Initalise
    cropped_cells = []
    masked_area = []

    # Make the time points 
    if len(roi_files) > 0: 
        for i in range(im_data.shape[0]): 
            # 
            time_point = i + 1
            
             # initalise 
            bg_file =  None
            cell_file = None
            # find connecting time points data.
            for j in range( len(roi_files) ):              
                # Find the underscores in the file name
                first_underscore = os.path.basename(roi_files[j]).find('_')
                second_underscore = os.path.basename(roi_files[j])[first_underscore + 1:].find('_') + first_underscore

                # Find the background file corresponding 
                # to the image file.
                if os.path.basename(roi_files[j])[0:second_underscore + 1] == 'bg_T' + str(time_point):
                    # Background image file
                    bg_file = roi_files[j]
                # Find the cell ROI file corresponding 
                # to the image file
                elif os.path.basename(roi_files[j])[0:second_underscore + 1] == 'cell_T' + str(time_point):
                    # Cell ROI
                    cell_file = roi_files[j]

            if cell_file == None or bg_file == None: 
                continue 
            else: 
                # create the polygon 
                cell_polygon  = np.array(roi_retrieval(cell_file))
                bg_polygon = np.array(roi_retrieval(bg_file))
                # Background intensity value
                background = np.mean(im_data[i, int(np.min(bg_polygon[:, 0])):int(np.max(bg_polygon[:,0])), 
                                int(np.min(bg_polygon[:,1])):int(np.max(bg_polygon[:,1]))])
                # Crop the cell out of the image
                cell_crop = create_crops_and_peak_find(cell_polygon, 
                                        im_data, i)
                # Duplicate cell crop 
                mask = np.array(cell_crop)
                # create a binary mask
                mask[mask<5*background] = 0
                mask[mask>=5*background] = 1
                # Create a set of labels from the binary mask. 
                labels = skimage.measure.label(mask)
                # Filtered labels 
                filtered_label = size_filter(labels)

                # Find peaks for each of the masks. 
                all_peaks = peaks_in_masks(filtered_label, cell_crop, 5*background)
                # Find masked area. 
                mask_area_pixels = len(np.where(filtered_label > 0)[0])
                # Convert filtered masks to a binary mask 
                binary_mask = np.array(filtered_label)
                binary_mask[binary_mask > 1] = 1
    
                # Perform the intensity measurements. 
                intensity_sum = np.sum(binary_mask * cell_crop)
                intensity_mean = np.mean( binary_mask * cell_crop)
                intensity_std = np.std( binary_mask * cell_crop )
                # Multiply cell crop by the binary mask. 
                masked_peaks = np.array(binary_mask * cell_crop)
                
                # Add in mask area. 
                if mask_area_pixels == 0:
                    mask_area_pixels = 1
                # Calculate intensity per unit area
                intensity_per_unit_area = intensity_sum / mask_area_pixels
            
                # Calculate the number of labels. 
                number_of_labels = np.max(filtered_label)
                # Calculate peaks per mask. 
                if number_of_labels == 0: 
                    peaks_per_mask = 0 
                    
                else: 
                    peaks_per_mask = len(all_peaks) / number_of_labels

                # Create image dictionary. 
                peak_dict = {'Image File': [file[0:-4]], 
                             'Time Point': [time_point],
                             'Mean Background Intensity': [np.round( background ) ],
                             'Number of peaks found in image': [len(all_peaks)], 
                             'Number of masked regions (regions over 5x background Intensity)': [int(number_of_labels)],
                             'Peaks per masked region': [np.round( peaks_per_mask, 3)],
                             'Total Mask Area (pixels)': [mask_area_pixels], 
                             'Intensity Sum for all pixels above 5x background Intensity': [int( intensity_sum )],
                             'mean Intensity for all pixels above 5x background Intensity': [np.round(intensity_per_unit_area, 3)], 
                             'Intensity Standard Deviation for all pixels above 5x background Intensity': [np.round( np.std(masked_peaks[masked_peaks>0]) )], 
                             'mean Intensity for all pixels above 5x background Intensity minus mean background instensity': [np.round(intensity_per_unit_area - background, 3)]
                            }
                # Save data. 
                peak_df = pd.concat([peak_df, pd.DataFrame(peak_dict)])
                # 
                time_point_folder = save_path + '/' +file[0:-4]  + '_2/'# + str(time_point) + '/'
                if os.path.exists(time_point_folder) == False:
                    os.makedirs(time_point_folder)

                save_figures(cell_crop, binary_mask, all_peaks, time_point_folder, file, time_point)

                

#     if len(cropped_cells) > 0: 
#         #
#         masked_area = np.array(masked_area)
#         #
#         tf.imwrite(save_path + '/' + file[:-4] + '_masks.tif', masked_area)


 42%|██████████████████████████████████▌                                                | 5/12 [00:24<00:23,  3.37s/it]C:\Users\rcorbyn\AppData\Local\anaconda3\envs\Ryan\lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\rcorbyn\AppData\Local\anaconda3\envs\Ryan\lib\site-packages\numpy\_core\_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\rcorbyn\AppData\Local\anaconda3\envs\Ryan\lib\site-packages\numpy\_core\_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
 50%|█████████████████████████████████████████▌                                         | 6/12 [00:32<00:29,  4.86s/it]C:\Users\rcorbyn\AppData\Local\anaconda3\envs\Ryan\lib\site-packages\numpy\_core\_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = 

In [ ]:
# peak_df.to_csv(os.path.dirname(folder) + '/' + '20250730_' + os.path.basename(
                    # folder) +'_inital_analysis.csv', index = False)

In [15]:
time_point_folder

'//data.beatson.gla.ac.uk/data/RCORBYN/User_Data/Current_Projects/20250611_Ximena_ReceptorSegregation/figures//XR199_SL1_TL_A1_Out_2/'